# LoRA Adaptörü Demo

`demo.ipynb` içinde eğitilmiş küçük `TinyQwen` modeline (`qwen3/tiny_qwen.pt`), `lora/` klasöründeki hazır kodla minik bir LoRA adaptörü takıp yapısını yazdırıyoruz. Adımlar (bkz. `lora/DEMO_ADAPTER_GUIDE.md`):

1. Modeli ve tokenizer'ı yükle.
2. `lora/` klasörünü import path'ine ekle, adapte edilebilir katmanları listele.
3. Adaptörü tak ve modelin yeni yapısını yazdır (`Linear` -> `LoRALinear`).
4. Eğitilebilir/toplam parametre oranına bak.
5. Tek bir harfle başlayan isimlerden minik bir veriyle sadece adaptörü eğit.
6. Adaptör açık/kapalıyken üretimi karşılaştır.
7. Adaptörü diske kaydet ve tensörlerinin şekillerini yazdır.

In [1]:
# 0) modeli ve tokenizer'ı yükle
import sys, torch
sys.path.insert(0, "qwen3")                      # match the architecture of the checkpoint
from model import TinyQwen
from tokenizer import CharTokenizer

ckpt = torch.load("qwen3/tiny_qwen.pt", map_location="cpu", weights_only=False)
tok = CharTokenizer(ckpt["chars"])
model = TinyQwen(ckpt["cfg"]); model.load_state_dict(ckpt["model"]); model.eval()
model

TinyQwen(
  (embed_tokens): Embedding(30, 32)
  (layers): ModuleList(
    (0-1): 2 x TransformerBlock(
      (input_layernorm): RMSNorm()
      (attn): Attention(
        (q_proj): Linear(in_features=32, out_features=32, bias=False)
        (k_proj): Linear(in_features=32, out_features=16, bias=False)
        (v_proj): Linear(in_features=32, out_features=16, bias=False)
        (o_proj): Linear(in_features=32, out_features=32, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (post_attention_layernorm): RMSNorm()
      (mlp): MLP(
        (gate_proj): Linear(in_features=32, out_features=64, bias=False)
        (up_proj): Linear(in_features=32, out_features=64, bias=False)
        (down_proj): Linear(in_features=64, out_features=32, bias=False)
      )
    )
  )
  (norm): RMSNorm()
  (lm_head): Linear(in_features=32, out_features=30, bias=False)
)

In [ ]:
# 1) lora/ klasöründeki hazır kodu import edelim
sys.path.insert(0, "lora")
from lora import LoRAConfig
from inject import (inject, set_adapters, merge_adapters, print_parameter_report,
                    print_linear_names, save_adapter)

# 2) hangi Linear katmanları adapte edilebilir?
print_linear_names(model)

In [ ]:
# 3) adaptörü tak: q_proj ve v_proj'a LoRA (orijinal makaledeki klasik seçim), r=4
lcfg = LoRAConfig(r=4, alpha=8.0, targets=("q_proj", "v_proj"))
adapted_layers = inject(model, lcfg, method="lora")
print("adapte edilen katmanlar:", adapted_layers)

model   # <- adaptörün yapısı: Linear yerine LoRALinear(base, lora_A, lora_B)

In [ ]:
# 4) eğitilebilir / toplam parametre oranı (adaptör ne kadar küçük?)
print_parameter_report(model)

In [ ]:
# 5) minik eğitim verisi: sadece 'z' ile başlayan isimler
letter = "z"
raw = open("data/temiz_isimler.txt", encoding="utf-8").read().split("\n")
names = [n for n in raw if n and n[0] == letter]
text = "\n" + "\n".join(names) + "\n"
data = torch.tensor(tok.encode(text), dtype=torch.long)
print(f"{len(names)} tane '{letter}' ile başlayan isim, {len(data)} karakter")

BLOCK_SIZE, BATCH_SIZE, STEPS, LR = 16, 64, 300, 5e-3
torch.manual_seed(0)

def get_batch(data):
    ix = torch.randint(len(data) - BLOCK_SIZE - 1, (BATCH_SIZE,))
    x = torch.stack([data[i:i + BLOCK_SIZE] for i in ix])
    y = torch.stack([data[i + 1:i + 1 + BLOCK_SIZE] for i in ix])
    return x, y

# sadece requires_grad=True olanlar (yani adaptör) optimize edilir, base donuk kalır
opt = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LR)
model.train()
for step in range(1, STEPS + 1):
    x, y = get_batch(data)
    _, loss = model(x, y)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 50 == 0 or step == 1:
        print(f"step {step:4d}  loss {loss.item():.4f}")
model.eval()

In [ ]:
# 6) adaptör açık/kapalı karşılaştırması
@torch.no_grad()
def sample(n=20):
    start = torch.full((n, 1), tok.eos_id, dtype=torch.long)
    out = model.generate(start, max_new_tokens=model.cfg.max_seq_len,
                         temperature=0.8, eos_id=tok.eos_id)
    names = [tok.decode(r[1:]).split("\n")[0] for r in out.tolist()]
    return [nm for nm in names if nm]

set_adapters(model, True)
print("adaptör AÇIK :", sample())
set_adapters(model, False)
print("adaptör KAPALI (orijinal model):", sample())
set_adapters(model, True)   # devam için tekrar açık bırak

In [ ]:
# 7) adaptörü diske kaydet ve içindeki tensörlerin gerçek "yapısını" yazdır
adapter_path = f"qwen3/adapter_{letter}.pt"
save_adapter(model, adapter_path, "lora", lcfg, arch="qwen3")

adapter_ckpt = torch.load(adapter_path, map_location="cpu", weights_only=False)
print(f"method={adapter_ckpt['method']}   cfg={adapter_ckpt['cfg']}\n")
for name, tensor in adapter_ckpt["adapter"].items():
    print(f"{name:30s} {tuple(tensor.shape)}")

## 8) LoRA matrisleri orijinal ağırlığa adım adım nasıl ekleniyor?

Her adapte edilmiş katmanda (örn. `layers.0.attn.q_proj`, `W`: `[32, 32]`) iki küçük matris var:

- **`lora_A`**  şekli `[r, in] = [4, 32]` — girdiyi 32 boyuttan 4 boyutlu "dar boğaza" sıkıştırır.
- **`lora_B`**  şekli `[out, r] = [32, 4]` — bu 4 boyutlu ara sonucu tekrar 32 boyuta genişletir.

Birleşme iki matematiksel olarak **birebir eşdeğer** yoldan anlaşılabilir:

**A) Ağırlık düzeyinde (merge — kalıcı birleştirme):**
```text
delta_W = scale * (B @ A)      # [32,4] @ [4,32] -> [32,32], W ile AYNI boyut
W_yeni  = W_orijinal + delta_W
```
(`scale = alpha / r`, klasik LoRA'da; burada `8.0 / 4 = 2.0`.)

**B) Girdi düzeyinde (forward — merge yapmadan, aynı sonucu üretir):**
```text
h      = x @ A.T            # [1,32] @ [32,4] -> [1,4]     (sıkıştır)
y_lora = scale * (h @ B.T)  # [1,4]  @ [4,32] -> [1,32]    (genişlet + ölçekle)
y      = (x @ W_orijinal.T) + y_lora
```

Aşağıdaki hücreler bunu `qwen3/adapter_z.pt` dosyasındaki (Adım 7'de kaydettiğimiz) **gerçek** `lora_A`/`lora_B` sayıları ve `tiny_qwen.pt` içindeki **gerçek** `W` ağırlığı ile gösteriyor — hiçbir şey uydurma değil, kendi eğittiğin adaptörün sayıları.

In [ ]:
# 8a) tek bir katman için: orijinal W + kaydedilmiş lora_A/lora_B -> delta_W -> W_yeni
W = ckpt["model"]["layers.0.attn.q_proj.weight"]          # [32, 32] -- hiç değişmedi, hâlâ orijinal
saved = torch.load(adapter_path, map_location="cpu", weights_only=False)
A = saved["adapter"]["layers.0.attn.q_proj.lora_A"]        # [4, 32]
B = saved["adapter"]["layers.0.attn.q_proj.lora_B"]        # [32, 4]
scale = saved["cfg"].alpha / saved["cfg"].r
print(f"scale = alpha/r = {saved['cfg'].alpha}/{saved['cfg'].r} = {scale}")

delta = scale * (B @ A)     # [32,32] -- W ile aynı boyutta bir "düzeltme tablosu"
W_new = W + delta

print("\nW[0, :5]      (orijinal)  :", W[0, :5])
print("delta[0, :5]  (LoRA eklentisi):", delta[0, :5])
print("W_new[0, :5]  (birleşmiş) :", W_new[0, :5])


In [ ]:
# 8b) gerçek bir girdiyle: "adım adım" (A ile sıkıştır, B ile genişlet) == "W_yeni ile tek seferde"
zid = tok.encode("z")[0]
x = ckpt["model"]["embed_tokens.weight"][zid].unsqueeze(0)   # [1, 32] -- 'z' harfinin embedding'i

y_base = x @ W.T                    # sadece orijinal ağırlıkla cevap
h      = x @ A.T                    # 32 -> 4 boyuta sıkıştır (LoRA'nın dar boğazı)
y_lora = scale * (h @ B.T)          # 4 -> 32 boyuta geri genişlet, ölçekle
y_stepwise = y_base + y_lora        # A) adım adım: base + düzeltme

y_merged = x @ W_new.T              # B) tek seferde: birleşmiş W_yeni ile direkt

print("h (sıkıştırılmış, r=4)      :", h)
print("y_base[:5]     (orijinal)   :", y_base[0, :5])
print("y_lora[:5]     (LoRA eklentisi):", y_lora[0, :5])
print("y_stepwise[:5] (base+lora)  :", y_stepwise[0, :5])
print("y_merged[:5]   (W_yeni ile) :", y_merged[0, :5])
print("\niki yol da birebir aynı mı?", torch.allclose(y_stepwise, y_merged, atol=1e-5))


## 9) `merge_adapters` — aynı birleştirmeyi canlı modelde, tüm katmanlarda birden yap

Yukarıda 8a/8b'de `delta_W = scale * (B @ A)` hesabını elle yaptık. `inject.py`
içindeki `merge_adapters(model)` fonksiyonu tam olarak bunu yapıyor — ama:

- **tek katman değil, adapte edilmiş HER katman için** (`model.modules()` içinde
  `merge` metodu olan her modülü bulup çağırıyor),
- her `LoRALinear.merge()` çağrısı kendi `base.weight`'ini `+= delta_weight()`
  ile kalıcı olarak günceller ve `enabled=False` yapar (bir daha `lora_A`/`lora_B`
  toplanmasın diye — yoksa düzeltme iki kere eklenmiş olurdu).

Yani `merge_adapters` sonrası model artık **sıradan bir `TinyQwen`** gibi
çalışır: `LoRALinear` sarmalayıcıları hâlâ orada durur ama içindeki `base`
ağırlığı zaten güncellendiği ve adapter yolu kapatıldığı için ekstra bir
maliyeti kalmaz.

⚠️ Bu hücre `model`'i **kalıcı olarak** değiştirir (bellekte). Adaptörü tekrar
ayrı ayrı açıp kapatmak istersen modeli yeniden yükleyip Adım 3'ü tekrar
çalıştırman gerekir. Diskteki `tiny_qwen.pt` ve `qwen3/adapter_z.pt`
dosyaları bundan etkilenmez.

In [ ]:
# 9a) merge ÖNCESİ: canlı modeldeki q_proj hâlâ orijinal W ve adapter ayrı duruyor
q_proj_layer = model.layers[0].attn.q_proj
print("merge öncesi tip           :", type(q_proj_layer).__name__)
print("merge öncesi enabled       :", q_proj_layer.enabled)
print("merge öncesi base.weight == orijinal W ?",
      torch.allclose(q_proj_layer.base.weight, W))

# logit'leri merge'den ÖNCE kaydet, karşılaştırma için
xb, _ = get_batch(data)
before_logits = model(xb)[0]


In [ ]:
# 9b) merge_adapters(model) -> HER LoRALinear kendi base.weight'ine kendi delta_weight()'ini gömer
merge_adapters(model)

print("merge sonrası enabled       :", q_proj_layer.enabled, "(artık kapalı: bir daha eklenmesin)")
print("merge sonrası base.weight == elle hesapladığımız W_new (8a) ?",
      torch.allclose(q_proj_layer.base.weight, W_new, atol=1e-5))

# 9c) tüm modelin çıktısı merge öncesi/sonrası birebir aynı mı?
after_logits = model(xb)[0]
max_diff = (before_logits - after_logits).abs().max().item()
print(f"\nmerge öncesi/sonrası logit farkı: {max_diff:.2e}  "
      f"({'OK, birebir aynı' if max_diff < 1e-4 else 'FARKLI!'})")
